# 2D FEP subtraction: diagnose the mismatch and write 2D output frames
Follow-up to the first trial. Uses the saved `ref_70_30.npz`, so the reference is **not** rebuilt.

**Questions**
1. Is the leftover FEP ring a *code/fit* problem or a *reference-matching* problem? (Test A vs Test B)
2. Does a merged reference built from non-hit frames near the test frame match better? (Test C)
3. Does masking the strong FEP reflection (5.45 to 6.0 degrees) clean up what is left? (Test D)
4. Write corrected 2D frames, ready for the diffraction sorter, and look at what the sorter would see.

Profiles below are **not baselined**, so negative lobes are visible (the MOR baseline in the first notebook hid them).

## Step 0: Paths and loading

In [ ]:
import os, glob, importlib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pyFAI, h5py
import fep2d_v2 as fm; importlib.reload(fm)

# ---- EDIT THESE (same as the first notebook) ----
OUT_DIR    = r"E:/AP39/.../Trial_2D_70_30/"
SAMPLE_DIR = r"E:/AP39/.../Run_seeded/02_V7_diff_sorting_output/diffraction/"
NONHIT_DIR = r"E:/AP39/.../Run_seeded/02_V7_diff_sorting_output/background/"
MASK_PATH  = r"E:/AP39/.../calib/mask.npy"
PONI_PATH  = r"E:/AP39/.../calib/calib.poni"
# --------------------------------------------------

sample_files = sorted(glob.glob(os.path.join(SAMPLE_DIR, "*.nxs")))
nonhit_files = sorted(glob.glob(os.path.join(NONHIT_DIR, "*.nxs")))
mask = np.load(MASK_PATH).astype(bool)
ai = pyFAI.load(PONI_PATH)
ref = fm.load_reference(os.path.join(OUT_DIR, "ref_70_30.npz"))
print(f"reference: {ref.label}, {ref.n_used} frames | samples {len(sample_files)} | non-hit {len(nonhit_files)}")

def profile(frame, m=mask, npt=2000):
    r = ai.integrate1d(frame.astype(np.float32), npt, unit=pyFAI.units.TTH_DEG,
                       radial_range=[1.0, 30.0], mask=m, correctSolidAngle=False, error_model=None)
    return np.asarray(r.radial), np.asarray(r.intensity)

def summarise(label, corr, info, m=mask):
    x, y = profile(corr, m)
    out = {"case": label, "scale": round(info["scale"], 3), "neg_frac": round(info["neg_frac"], 3)}
    for lo, hi in [(1, 5), (5, 7), (7, 30)]:
        s = (x >= lo) & (x <= hi)
        out[f"mean {lo}-{hi}"] = round(float(y[s].mean()), 2)
        out[f"std {lo}-{hi}"] = round(float(y[s].std()), 2)
    return out, x, y

def get_frame(path, idx=0):
    for p, i, fr in fm.iter_frames([path]):
        if i == idx: return fr

## Test A: noise floor (leave-one-out)
Takes one frame that went **into** the reference, removes it from the reference (exactly), and subtracts.
This must come out as pure noise. If it does not, the problem is in the code or the scale fit.

In [ ]:
p0, i0 = ref.sources[0]
frame_A = get_frame(p0, i0)
ref_loo = fm.leave_one_out_reference(ref, frame_A)
corr_A, info_A = fm.subtract_frame(frame_A, ref_loo, mask)
sumA, xA, yA = summarise("A: reference frame vs its own leave-one-out", corr_A, info_A)
print(sumA)

## Test B: the original null test (non-hit frame vs 70:30 reference)
This is what you saw as the blue and red rings.
Choose `TEST_IDX` so the frame is a clean solution-filled non-hit frame from the seeded run.

In [ ]:
TEST_IDX = len(nonhit_files) // 2          # EDIT: index of the non-hit test frame
test_path = nonhit_files[TEST_IDX]
frame_B = get_frame(test_path)
corr_B, info_B = fm.subtract_frame(frame_B, ref, mask)
sumB, xB, yB = summarise("B: non-hit vs merged 70:30", corr_B, info_B)
print(os.path.basename(test_path), sumB)

## Test C: merged reference built from non-hit frames close in time to the test frame
Still a **merged** reference (many frames averaged), but matched to the run. The test frame itself is excluded.
This separates "70:30 solvent does not match this run" from "the method cannot work".

In [ ]:
HALF = 50                                    # frames either side of the test frame
lo, hi = max(0, TEST_IDX - HALF), min(len(nonhit_files), TEST_IDX + HALF + 1)
nh_paths = [f for f in nonhit_files[lo:hi] if f != test_path]
ref_nh = fm.build_reference(nh_paths, mask=mask, label="merged non-hit (local)")
print(f"non-hit reference: {ref_nh.n_used} frames used, {ref_nh.n_rejected} rejected")
fm.save_reference(ref_nh, os.path.join(OUT_DIR, "ref_nonhit_local.npz"))
corr_C, info_C = fm.subtract_frame(frame_B, ref_nh, mask)
sumC, xC, yC = summarise("C: non-hit vs merged non-hit (local)", corr_C, info_C)
print(sumC)

## Test D: mask the strong FEP reflection, as in your 1D workflow
Adds an annulus mask (default 5.45 to 6.0 degrees 2theta). Masked pixels are left out of the scale fit, set to 0 in the corrected frame, and ignored on integration.

In [ ]:
EXTRA_MASK_TTH = [(5.40, 6.00)]              # same additional mask as your 1D workflow
mask2 = mask | fm.tth_annulus_mask(ai, mask.shape, EXTRA_MASK_TTH)
print(f"extra mask removes {100*(mask2.mean()-mask.mean()):.2f}% of the detector")

corr_D, info_D = fm.subtract_frame(frame_B, ref, mask2)
sumD, xD, yD = summarise("D: non-hit vs 70:30, FEP annulus masked", corr_D, info_D, mask2)
corr_E, info_E = fm.subtract_frame(frame_B, ref_nh, mask2)
sumE, xE, yE = summarise("E: non-hit vs non-hit ref, FEP annulus masked", corr_E, info_E, mask2)

# Same SCALING REGION as your 1D workflow (app default 2-4 deg: FEP only, no sample peaks)
SCALE_TTH = (2.0, 4.0)                       # EDIT to the scaling region you actually use
fit_reg = fm.tth_region(ai, mask.shape, SCALE_TTH)
print(f"scaling region {SCALE_TTH} deg = {int(fit_reg.sum())} pixels")
corr_F, info_F = fm.subtract_frame(frame_B, ref, mask2, fit_region=fit_reg)
sumF, xF, yF = summarise("F: 70:30, masked, scale from scaling region", corr_F, info_F, mask2)
corr_G, info_G = fm.subtract_frame(frame_B, ref_nh, mask2, fit_region=fit_reg)
sumG, xG, yG = summarise("G: non-hit ref, masked, scale from scaling region", corr_G, info_G, mask2)

table = pd.DataFrame([sumA, sumB, sumC, sumD, sumE, sumF, sumG])
print(table.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
for name, x, y in [("A leave-one-out", xA, yA), ("B 70:30", xB, yB), ("C non-hit ref", xC, yC),
                   ("D 70:30 + mask", xD, yD), ("E non-hit ref + mask", xE, yE),
                   ("F 70:30 + mask + scale region", xF, yF), ("G non-hit + mask + scale region", xG, yG)]:
    ax[0].plot(x, y, lw=1, label=name); ax[1].plot(x, y, lw=1, label=name)
ax[0].set_title("Corrected null frame, 1D profile (not baselined)"); ax[1].set_xlim(4, 8)
ax[1].set_title("Zoom on the FEP reflection region")
for a in ax: a.axhline(0, color="k", lw=0.5); a.set_xlabel("2θ (°)"); a.set_ylabel("mean counts / pixel"); a.grid(alpha=0.3)
ax[0].legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "diag_null_profiles.png"), dpi=150); plt.show()

fig, ax = plt.subplots(1, 4, figsize=(24, 5.5))
for a, (t, c, m_) in zip(ax, [("B: 70:30 reference", corr_B, mask), ("C: non-hit reference", corr_C, mask),
                              ("D: 70:30 + FEP annulus mask", corr_D, mask2),
                              ("F: 70:30 + mask + scale region", corr_F, mask2)]):
    lim = 4 * np.nanstd(corr_A[~mask])             # colour scale = noise floor from Test A
    a.imshow(np.where(m_, np.nan, c), origin="lower", vmin=-lim, vmax=lim, cmap="RdBu_r"); a.set_title(t)
plt.suptitle("Residuals on the SAME colour scale (±4 x noise floor from Test A)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "diag_null_residuals.png"), dpi=150); plt.show()

### How to read Steps A to E
- **A is noise only.** Its `std` values are your noise floor and its means should be about 0. Anything noticeably worse in A means a code or fit problem.
- **B much worse than A** means the 70:30 reference does not match this frame (composition, dissolved drug, or tube position), not a coding bug.
- **C better than B** means the merged-reference approach works, but the reference needs to match the run. A non-hit merged reference can then be your fallback.
- **D/E** show whether masking the strong reflection leaves a clean frame. For sorting this is usually enough. For quantitative work, avoid masking real peaks.
- **F/G** fit the scale only in your 1D **scaling region** (2θ range with FEP only, no sample peaks), exactly as the app does, instead of over the whole detector. If F is steadier and cleaner than D, use the scaling region in the final pipeline.

## Step 6: What the sorter would see (a diffraction frame, corrected)
Four panels:
1. **Raw frame.**
2. **Background-subtracted.** The colour scale is clipped to the 1st to 99th percentile. Small spots are easy to miss here because the large ring-shaped residuals dominate the scale.
3. **Significance** (corrected / expected noise).
4. **Spot view**: the median of each radial ring is removed (anything smooth around a ring goes: FEP leftovers, solvent halo, reference mismatch), then divided by the noise. Block-maximum display keeps single-pixel spots visible. This is the kind of input a feature extractor would use.

In [ ]:
import time
from scipy import ndimage

REF_TO_USE = ref            # or ref_nh once Test C has been checked
FIT_REGION_TO_USE = fit_reg # your 1D-style scaling region; set to None to fit all pixels
MASK_TO_USE = mask2         # mask2 = detector mask + FEP annulus; use `mask` for no extra mask
SHOW_IDX = 0                # which diffraction frame to look at

p = sample_files[SHOW_IDX]; fr = get_frame(p)
t0 = time.time()
corr, info = fm.subtract_frame(fr, REF_TO_USE, MASK_TO_USE, fit_region=FIT_REGION_TO_USE)
t_sub = time.time() - t0
sigma = np.sqrt(np.maximum(fr, 0) + info["scale"] ** 2 * REF_TO_USE.var_of_mean + 1e-9)
z = corr / sigma

ringmap = fm.build_ring_map(ai, mask.shape, MASK_TO_USE)     # build once per geometry/mask
t0 = time.time(); hp = fm.ring_median_subtract(corr, ringmap); t_ring = time.time() - t0
valid_ = ~MASK_TO_USE
spot_z = hp / sigma
spot_z = spot_z / (1.4826 * np.median(np.abs(spot_z[valid_])))   # empirical noise units
labels_, n_clusters = ndimage.label(spot_z > 5)
print(os.path.basename(p), {k: round(v, 3) for k, v in info.items()})
print(f"time per frame: subtraction {t_sub:.2f} s, ring-median {t_ring:.2f} s")
print(f"pixels above 5 sigma in spot view: {int((spot_z > 5).sum())} in {n_clusters} clusters")

def block_max(a, b=4):
    h, w = (a.shape[0] // b) * b, (a.shape[1] // b) * b
    return a[:h, :w].reshape(h // b, b, w // b, b).max(axis=(1, 3))

fig, ax = plt.subplots(1, 4, figsize=(30, 7.5))
v = fr[valid_]
ax[0].imshow(np.where(MASK_TO_USE, np.nan, fr), origin="lower", vmin=0, vmax=np.percentile(v, 99.5)); ax[0].set_title("Raw frame")
lo_, hi_ = np.percentile(corr[valid_], [1, 99])
ax[1].imshow(np.where(MASK_TO_USE, np.nan, corr), origin="lower", vmin=lo_, vmax=hi_, cmap="RdBu_r"); ax[1].set_title("Background-subtracted (1-99 percentile)")
ax[2].imshow(np.where(MASK_TO_USE, np.nan, z), origin="lower", vmin=-6, vmax=6, cmap="RdBu_r"); ax[2].set_title("Significance (sigma)")
ax[3].imshow(block_max(np.where(MASK_TO_USE, 0, spot_z)), origin="lower", vmin=0, vmax=12, cmap="magma",
             extent=[0, spot_z.shape[1], 0, spot_z.shape[0]]); ax[3].set_title("Spot view: ring median removed (block max, noise units)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "diag_sorter_view.png"), dpi=150); plt.show()

## Step 7: Write corrected 2D frames (the output for the sorter)
Writes one corrected file per input frame into `corrected_2D/` with the **original file name** and the same internal layout, so your existing scripts can read them. Both hits and non-hits are written, because the classifier needs both classes.
Corrected frames are float32, negative values are kept, masked pixels are 0, and the settings used are stored as attributes on the dataset.
Check that your sorting scripts read float32 and cope with negative values.

In [ ]:
WRITE_N = 20                # frames per class to write; the sorter needs BOTH hits and non-hits
write_files = sample_files[:WRITE_N] + nonhit_files[:WRITE_N]
out2d = os.path.join(OUT_DIR, "corrected_2D")
xy_out = os.path.join(OUT_DIR, "corrected_xy")
# suffix="" keeps the ORIGINAL file names, so the existing triage/sorting scripts can be
# pointed at this folder unchanged. .xy files (2000 pts, FEP range interpolated across)
# go to a separate folder in the app's format for the pattern viewer.
rows = fm.process_run(write_files, REF_TO_USE, out2d, mask=MASK_TO_USE, ai=ai, xy_dir=xy_out,
                      write_nxs=True, suffix="", npt=2000, interp_ranges=EXTRA_MASK_TTH,
                      fit_region=FIT_REGION_TO_USE)
print(f"wrote {len(rows)} corrected frames to {out2d} and .xy to {xy_out}")
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "subtraction_diagnostics_all.csv"), index=False)

# read-back check
chk = sorted(glob.glob(os.path.join(out2d, "*.nxs")))[0]
with h5py.File(chk, "r") as f:
    path = fm._detector_path(f); d = f[path]
    a = d[()]
    print(os.path.basename(chk), "| dataset:", path, "| shape:", d.shape, "| dtype:", d.dtype,
          "| min/max:", float(a.min()), float(a.max()))
    print({k: (v if not isinstance(v, bytes) else v.decode()) for k, v in d.attrs.items()
           if k in ("processing", "reference_label", "reference_n_frames", "scale_factor")})

## Step 8: The deciding test, baselined patterns on real diffraction frames vs your 1D result
The null-frame profiles in Tests A to G are not the final product. What matters is the pattern after your steps: **mask range filled by a straight line, MOR baseline (half-window 4), normalised to max = 1**.
This overlays four 2D options (and optionally your 1D-corrected pattern) on the same diffraction frames.
Metric: peak-to-noise = the normalised maximum (1.0, taken outside the masked range) divided by the standard deviation in the diagnostic region. Higher is cleaner.

In [ ]:
from pybaselines import Baseline
HW = 4
DIAG = (7.0, 9.0)                 # EDIT: your diagnostic region (deg)
ONE_D_DIR = r""                   # optional: folder with your 1D-corrected .xy files; "" to skip
CMP_FRAMES = sample_files[:6]     # EDIT: frames that contain real crystal peaks

combos = {
    "70:30 + mask":                  (ref,    None),
    "70:30 + mask + scale region":   (ref,    fit_reg),
    "non-hit + mask":                (ref_nh, None),
    "non-hit + mask + scale region": (ref_nh, fit_reg),
}

def finish(x, y):
    # fill masked 2theta ranges by straight line, MOR baseline, normalise to max=1 outside the mask
    for lo, hi in EXTRA_MASK_TTH:
        y = fm.interpolate_gap(x, y, lo, hi)
    yb = y - Baseline(x_data=x).mor(y, half_window=HW)[0]
    inmask = np.zeros(len(x), bool)
    for lo, hi in EXTRA_MASK_TTH:
        inmask |= (x >= lo) & (x <= hi)
    return yb / max(yb[~inmask].max(), 1e-9)

rows = []
for p, i, fr in fm.iter_frames(CMP_FRAMES):
    stem = os.path.splitext(os.path.basename(p))[0]
    curves = {}
    for name, (rf, reg) in combos.items():
        corr, info = fm.subtract_frame(fr, rf, mask2, fit_region=reg)
        x, y = profile(corr, mask2)
        curves[name] = (x, finish(x, y))
    if ONE_D_DIR:
        cand = glob.glob(os.path.join(ONE_D_DIR, f"{stem}*.xy"))
        if cand:
            d = np.loadtxt(cand[0], skiprows=1)
            curves["your 1D result"] = (d[:, 0], finish(d[:, 0], d[:, 1]))
    fig, ax = plt.subplots(1, 2, figsize=(17, 4.5))
    for name, (x, yn) in curves.items():
        ax[0].plot(x, yn, lw=1, label=name); ax[1].plot(x, yn, lw=1, label=name)
        m = (x >= DIAG[0]) & (x <= DIAG[1])
        rows.append(dict(frame=stem, option=name, diag_std=round(float(yn[m].std()), 4),
                         peak_to_noise=round(1.0 / max(float(yn[m].std()), 1e-9), 1)))
    ax[0].set_title(f"{stem}: baselined, normalised"); ax[1].set_ylim(-0.05, 0.3); ax[1].set_title("zoom on the low end")
    for a in ax: a.axvspan(*DIAG, color="gray", alpha=0.1); a.set_xlabel("2θ (°)"); a.grid(alpha=0.3)
    ax[0].legend(fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"cmp_{stem}.png"), dpi=150); plt.show()

cmp_table = pd.DataFrame(rows).pivot(index="frame", columns="option", values="peak_to_noise")
print("Peak-to-noise (higher = cleaner):"); print(cmp_table.to_string())
print("\nMean over frames:"); print(cmp_table.mean().round(1).to_string())